# Banco de Dados - Plantas de Veiculos Eletricos

Este notebook coleta, valida e salva GHI diario para 10 localidades usando o produto oficial NLR/NSRDB GOES Aggregated PSM v4.

O periodo solicitado e 2019-2025. A rotina consulta a API e usa somente anos historicos efetivamente publicados, sem gerar ou completar dados sinteticos.

## Ambiente e funcoes de coleta

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "cadernos_jupyter":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

from codigo_fonte.preprocessamento import (
    coletar_ghi_nrel,
    consultar_anos_disponiveis_nsrdb,
)
from codigo_fonte.localidades_ev import LOCALIDADES_EV

## Credenciais

O arquivo `.env` da raiz deve definir `NREL_API_KEY` e `NREL_EMAIL`. Os nomes historicos foram mantidos por compatibilidade com o projeto.

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

key = os.getenv("NREL_API_KEY")
email = os.getenv("NREL_EMAIL")

if not key or not email:
    raise ValueError("Preencha NREL_API_KEY e NREL_EMAIL no arquivo .env.")

print("Credenciais NLR carregadas do arquivo .env.")

## Localidades

In [ ]:
localidades = LOCALIDADES_EV

display(pd.DataFrame(localidades))

## Disponibilidade e coleta oficial

In [ ]:
INTERVALO_MINUTOS = 60
ANO_INICIAL = 2019
ANO_FINAL_SOLICITADO = 2025

referencia = localidades[0]
anos_disponiveis = consultar_anos_disponiveis_nsrdb(
    referencia["lat"], referencia["lon"], key
)
anos_no_periodo = [
    ano for ano in anos_disponiveis
    if ANO_INICIAL <= ano <= ANO_FINAL_SOLICITADO
]
if not anos_no_periodo:
    raise RuntimeError("Nenhum ano solicitado esta disponivel no NSRDB.")

ANO_FINAL_COLETA = max(anos_no_periodo)
if ANO_FINAL_COLETA < ANO_FINAL_SOLICITADO:
    print(
        f"AVISO: {ANO_FINAL_SOLICITADO} ainda nao foi publicado no produto "
        f"GOES Aggregated PSM v4. A coleta oficial terminara em {ANO_FINAL_COLETA}."
    )

output_dir = PROJECT_ROOT / "dados" / "brutos" / "localidades_ev"
dados_localidades = {}

for local in localidades:
    arquivo = output_dir / (
        local["nome"].lower().replace(" ", "_").replace("-", "_") + ".csv"
    )
    print(f"Coletando {local['nome']} -> {arquivo.name}")
    dados_localidades[local["nome"]] = coletar_ghi_nrel(
        lat=local["lat"],
        lon=local["lon"],
        city=local["nome"],
        pais=local["pais"],
        endereco=local["endereco"],
        fonte_localidade=local["fonte_localidade"],
        fonte_coordenadas=local["fonte_coordenadas"],
        metodo_coordenadas=local["metodo_coordenadas"],
        osm_elemento=local["osm_elemento"],
        start_year=ANO_INICIAL,
        end_year=ANO_FINAL_COLETA,
        inter=INTERVALO_MINUTOS,
        output_path=arquivo,
    )

print(f"Coleta finalizada e salva em {output_dir}.")

## Validacao dos dados coletados

In [ ]:
ghi_total = pd.concat(dados_localidades.values(), ignore_index=True)
datas_esperadas = pd.date_range(
    f"{ANO_INICIAL}-01-01", f"{ANO_FINAL_COLETA}-12-31", freq="D"
)

cobertura = (
    ghi_total.groupby("localidade")
    .agg(
        registros=("data", "size"),
        inicio=("data", "min"),
        fim=("data", "max"),
        ghi_minimo=("ghi", "min"),
        ghi_medio=("ghi", "mean"),
        ghi_maximo=("ghi", "max"),
    )
    .sort_values("ghi_medio", ascending=False)
)

if not (cobertura["registros"] == len(datas_esperadas)).all():
    raise ValueError("Uma ou mais localidades estao com cobertura diaria incompleta.")
if not ghi_total["ghi"].between(0, 500).all():
    raise ValueError("Foram encontrados valores fora da unidade diaria W/m2.")

display(cobertura)
display(ghi_total.head())
display(ghi_total.tail())

display(
    ghi_total[[
        "fonte_dados", "produto_dados", "versao_dados",
        "endpoint_api", "intervalo_minutos", "agregacao", "unidade_ghi",
    ]].drop_duplicates()
)

## Series mensais para visualizacao

In [ ]:
for local in localidades:
    nome = local["nome"]
    mensal = (
        dados_localidades[nome]
        .set_index("data")[["ghi"]]
        .resample("ME")
        .mean()
    )

    fig, ax = plt.subplots(figsize=(8, 3))
    mensal["ghi"].plot(ax=ax, color="#FFBF00", linewidth=2, label="GHI")
    ax.grid(True, alpha=0.5, ls="-.")
    ax.set_xlabel("Ano")
    ax.set_ylabel("GHI medio diario [W/m2]")
    ax.set_title(nome)
    plt.tight_layout()
    plt.show()

## Comparativo entre localidades

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for local in localidades:
    nome = local["nome"]
    mensal = (
        dados_localidades[nome]
        .set_index("data")[["ghi"]]
        .resample("ME")
        .mean()
    )
    ax.plot(mensal.index, mensal["ghi"], linewidth=1.5, alpha=0.85, label=nome)

ax.grid(True, alpha=0.5, ls="-.")
ax.set_xlabel("Ano")
ax.set_ylabel("GHI medio diario [W/m2]")
ax.set_title("Comparativo de GHI - plantas de veiculos eletricos")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()